<a href="https://colab.research.google.com/github/karkhutmaria/KarkhutM_topical_compling_questions/blob/main/KarkhutM__05_homework_harness_benchmarks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Задание 1 (8 баллов + 0.5)

Скачайте вот этот датасет - ("MERA-evaluation/MERA", "multiq"). Так как обучать ничего не придется возьмите train split. Возьмите оттуда 25 примеров (случайных, а не первых).

Преобразуйте колонку instruction, заменив переменные вроде `{support_text}` на реальные значения из других колонок, чтобы получился готовый промпт (инпут для модели).

#### Метрика (2 балла)

Напишите две метрики для оценки правильных ответов: 1) exact match и 2) F1 по леммам. Для первой просто добавьте базовую предобработку вроде удаления пунктуации и регистра и просто сопоставьте строки между собой. Для второй к базовой предобработке добавьте токенизацию и лемматизацию, а поверх лемм рассчитайте F1 метрику (подсказка - precision можно рассчитать посчитав какой процент лемм в предсказанном ответе совпадает с правильным ответом).

Рассчитайте метрики на простом бейзлайне (например, можете попробовать взять самый частотный биграмм из промпта в качестве ответа). Можно получить +0.5 балла за неочевидный ненулевой бейзлайн.

#### Эксперимент 1 (2 балла)
Возьмите какую-нибудь не очень большую модель с huggingface (например, `Qwen/Qwen3.5-0.8B`, но можно и любую другую, которая умеет отвечать на инструкции и поддерживает thinking). Преобразуйте промпты из датасета в формат `messages = [{"role": "user", "content": prompt}]` с помощью чат темплейта этой модели.
Сгенерируйте ответы модели на полученных данных (25 примеров). На этом шаге не используйте thinking. Поставьте max_new_tokens на не очень большое значения, чтобы генерация не занимала слишком много времени.
Оцените качество ответов с помощью двух метрик.

#### Эксперимент 2 (1 балла)
К первому эксперименту добавьте какой-то system prompt, который по вашему должен помочь модели справится с результатом. Можете проанализировать ошибки из первого задания и написать инструкции на основе этого.
Сгенерируйте ответы с точно такими же параметрами как и в эксперименте 1.
Оцените качество ответов с помощью двух метрик.

#### Эсперимент 3 (3 балла)
На этом шаге включите thinking в модели и сгенерируйте предсказания (system prompt вы можете использовать по вашему желанию).
На этом шаге у вас будет несколько сложностей, которые нужно решить:
1) генерацию модели нельзя использовать напрямую, сначала нужно отделить thinking блок от самого ответа. Добавьте пост обработку
2) thinking может занять достаточно много токенов и его нужно ограничить, но при этом вам нужно, чтобы модель выдала и сам ответ, а не только незаконченные размышлений. Выберите max_new_tokens с рассчетом на (thinking_tokens + answer_tokens). Сначала запустите первую генерацию с thinking_tokens, затем проверьте, что в ответе содержится </think> тэг, который означает, что модель закончила думать (еще удобно использовать `stop_strings` в huggingface generation параметрах, чтобы заканчивать как только появился нужный токен). И если тэг есть, то просто запустите вторую генерацию с max_new_tokens (как продолжение уже накопившегося) или же поставьте </think> тэг вручную (обрубив размышление) и запустите вторую генерацию c answer_tokens.


Загружаем все необходимые библиотеки

In [18]:
!pip install datasets pandas numpy matplotlib scikit-learn math-verify pymorphy3 torch
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import datasets
from datasets import load_dataset, get_dataset_config_names
import random
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import pymorphy3

Загружаем датасет и выводим 25 случайных примеров

In [19]:
multiq = load_dataset("MERA-evaluation/MERA", "multiq", split="train")

In [20]:
random.seed(42)
indices = random.sample(range(len(multiq)), 25)
samples = [multiq[i] for i in indices]

def build_prompt(example):
  instruction = example["instruction"]
  inputs = example["inputs"]

  prompt = instruction.format(
      support_text=inputs.get("support_text", ""),
      text=inputs.get("text", ""),
      question=inputs.get("question", "")
  )
  return prompt

def extract_gold(example):
  outputs = example["outputs"]
  if isinstance(outputs, list) and len(outputs) > 0:
      labels = [o["label"] for o in outputs]
      return labels[0] if len(labels) == 1 else " ".join(labels)
  return str(outputs)

prompts = [build_prompt(ex) for ex in samples]
golds = [extract_gold(ex) for ex in samples]

for prompt in prompts:
  print("Промпт:\n", prompt[:500], "...")
  print("\n")

Промпт:
 Прочитайте два текста.
1. Франческо Корнер, или Корнаро  ( 6 марта 1585, Венеция — 5 июня 1656, там же) — 101-й венецианский дож, третий по счёту из знатного венецианского рода Корнаро. Избран на должность 17 мая 1656 года, однако пробыл там всего лишь 18 дней. Его короткое правление не было отмечено какими-либо событиями, однако стало самым коротким в истории Венецианской республики.\n\nБиография.\nВыходец из богатой и влиятельной семьи, Франческо сделал невероятную политическую карьеру. Лишь е ...


Промпт:
 Даны два текста:
Текст 1: Зиновий Биртман — музыкальная мистификация, появившаяся в 2014 (первые видео на Youtube-канале появились в это время) году и ставшая основой для группы "Биртман": вымышленный советский автор-исполнитель.\n\nБиография.\nСогласно легенде, Зиновий Аркадьевич Биртман родился 12 ноября 1934 года в семье работника НКВД Аркадия Леонидовича Биртмана и преподавателя женской гимназии Нины Иосифовны Струцовской. После войны и развода родителей остался в Тю

Пишем метрики и функции предобработки примеров

In [21]:
morph = pymorphy3.MorphAnalyzer()

def normalize_text(text):
  text = text.lower()
  text = text.translate(str.maketrans("", "", string.punctuation + "«»—–…"))
  text = re.sub(r"\s+", " ", text).strip()
  return text

def exact_match(pred, gold):
  return int(normalize_text(pred) == normalize_text(gold))

def lemmatize(text):
  text = normalize_text(text)
  tokens = text.split()
  lemmas = [morph.parse(token)[0].normal_form for token in tokens]
  return lemmas

def f1_lemmas(pred, gold):
  pred_lemmas = lemmatize(pred)
  gold_lemmas = lemmatize(gold)
  if not pred_lemmas or not gold_lemmas:
    return 0.0

  common = Counter(pred_lemmas) & Counter(gold_lemmas)
  num_same = sum(common.values())
  if num_same == 0:
    return 0.0
  precision = num_same / len(pred_lemmas)
  recall = num_same / len(gold_lemmas)
  f1 = 2 * precision * recall / (precision + recall)
  return f1

In [22]:
def baseline_most_frequent_bigram(prompt):
  text = prompt.lower()
  words = re.findall(r"\w+", text)
  if len(words) < 2:
    return words[0] if words else ""
  bigrams = list(zip(words, words[1:]))
  most_common = Counter(bigrams).most_common(1)[0][0]
  return " ".join(most_common)

baseline_preds = [baseline_most_frequent_bigram(p) for p in prompts]

In [23]:
baseline_exact_match = sum(exact_match(p, g) for p, g in zip(baseline_preds, golds)) / len(golds)
baseline_f1 = sum(f1_lemmas(p, g) for p, g in zip(baseline_preds, golds)) / len(golds)

print(f"Бейзлайн по самому частотному биграмму:")
print(f"Exact Match: {baseline_exact_match:.2f}")
print(f"F1: {baseline_f1:.4f}")
print(f"Примеры предсказаниий: {baseline_preds[0]}\n{baseline_preds[1]}\n{baseline_preds[2]}\n{baseline_preds[3]}")

Бейзлайн по самому частотному биграмму:
Exact Match: 0.04
F1: 0.0800
Примеры предсказаниий: франческо корнер
зиновий биртман
ара акопович
река в


Загружаем модель

In [24]:
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
  print(f"GPU: {torch.cuda.get_device_name(0)}")
  print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
  cap = torch.cuda.get_device_capability(0)
  print(f"Compute capability: {cap}")
  dtype = torch.bfloat16 if cap[0] >= 8 else torch.float16
  device_map = "cuda"
else:
  dtype = torch.float32
  device_map = "cpu"
  print("WARNING: No GPU detected — generation will be very slow!")

model_name = "Qwen/Qwen3.5-0.8B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=dtype,
    device_map=device_map,
    attn_implementation="sdpa",
)
model.eval()

print(f"\nModel device: {next(model.parameters()).device}")
print(f"Model dtype:  {next(model.parameters()).dtype}")

_ = model.generate(**tokenizer("hi", return_tensors="pt").to(model.device), max_new_tokens=5)
print("Warmup done.")

CUDA available: True
GPU: Tesla T4
Memory: 15.6 GB
Compute capability: (7, 5)


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]


Model device: cuda:0
Model dtype:  torch.float16
Warmup done.


In [25]:
def generate_batch(prompts, system_prompt=None, enable_thinking=False,
                   max_new_tokens=64, thinking_budget=256, answer_budget=64,
                   batch_size=25):

    texts = []
    for p in prompts:
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": p})
        texts.append(tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=enable_thinking,
        ))

    responses = [None] * len(texts)
    gen_kwargs = dict(do_sample=False, pad_token_id=tokenizer.pad_token_id)

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to(model.device)
        input_len = inputs["input_ids"].shape[1]

        if not enable_thinking:
            with torch.inference_mode():
                out = model.generate(**inputs, max_new_tokens=max_new_tokens, **gen_kwargs)
            for i in range(out.shape[0]):
                responses[start + i] = tokenizer.decode(
                    out[i][input_len:], skip_special_tokens=True
                ).strip()
            continue

        with torch.inference_mode():
            out = model.generate(
                **inputs,
                max_new_tokens=thinking_budget,
                stop_strings=["</think>"],
                tokenizer=tokenizer,
                **gen_kwargs,
            )
        need_second_texts, need_second_idx = [], []
        for i in range(out.shape[0]):
            thinking_part = tokenizer.decode(out[i][input_len:], skip_special_tokens=False)
            if "</think>" in thinking_part:
                answer = thinking_part.split("</think>")[-1].strip()
                answer = tokenizer.decode(
                    tokenizer.encode(answer, add_special_tokens=False),
                    skip_special_tokens=True,
                ).strip()
                if answer:
                    responses[start + i] = answer
                    continue
                need_second_idx.append(start + i)
                need_second_texts.append(texts[start + i] + thinking_part)
            else:
                need_second_idx.append(start + i)
                need_second_texts.append(texts[start + i] + thinking_part + "</think>")


        if need_second_texts:
            inputs2 = tokenizer(
                need_second_texts, return_tensors="pt", padding=True
            ).to(model.device)
            input_len2 = inputs2["input_ids"].shape[1]
            with torch.inference_mode():
                out2 = model.generate(
                    **inputs2, max_new_tokens=answer_budget, **gen_kwargs
                )
            for j in range(out2.shape[0]):
                responses[need_second_idx[j]] = tokenizer.decode(
                    out2[j][input_len2:], skip_special_tokens=True
                ).strip()

    return responses

Эксперимент 1

In [26]:
import time

t0 = time.time()
preds_exp1 = generate_batch(prompts, system_prompt=None, enable_thinking=False,
                            max_new_tokens=64)
print(f"Прошло времени: {time.time() - t0:.1f}s")

em_exp1 = sum(exact_match(p, g) for p, g in zip(preds_exp1, golds)) / len(golds)
f1_exp1 = sum(f1_lemmas(p, g) for p, g in zip(preds_exp1, golds)) / len(golds)

print(f"\nЭксперимент 1:")
print(f"Exact Match: {em_exp1:.2f}")
print(f"F1: {f1_exp1:.2f}")
print(f"\nПример: {preds_exp1[0]}")
print(f"Правильный ответ: {golds[0]}")
print(f"\nПример: {preds_exp1[1]}")
print(f"Правильный ответ: {golds[1]}")

Прошло времени: 10.9s

Эксперимент 1:
Exact Match: 0.12
F1: 0.36

Пример: Венечский совет
Правильный ответ: Большой совет

Пример: Руслан Николаевич Кухарук
Правильный ответ: Руслан Николаевич Кухарук


Эксперимент 2

In [31]:
SYSTEM_PROMPT = ("""
    Ты — эксперт по русскому языку и логике. Внимательно прочитай текст и вопрос.
    Отвечай кратко и по делу: только правильным ответом, без лишних пояснений и лишних слов. Ответ из нескольких слов должен быть разделен пробелом.
""")

t0 = time.time()
preds_exp2 = generate_batch(prompts, system_prompt=SYSTEM_PROMPT, enable_thinking=False,
                            max_new_tokens=64)
print(f"Прошло времени: {time.time() - t0:.1f}s")

em_exp2 = sum(exact_match(p, g) for p, g in zip(preds_exp2, golds)) / len(golds)
f1_exp2 = sum(f1_lemmas(p, g) for p, g in zip(preds_exp2, golds)) / len(golds)

print(f"\nЭксперимент 2:")
print(f"Exact Match: {em_exp2:.4f}")
print(f"F1 (леммы):  {f1_exp2:.4f}")
print(f"\nПример: {preds_exp2[0]}")
print(f"Правильный ответ: {golds[0]}")

Прошло времени: 6.9s

Эксперимент 2:
Exact Match: 0.1200
F1 (леммы):  0.3541

Пример: Венечский совет
Правильный ответ: Большой совет


Эксперимент 3

In [34]:
t0 = time.time()
preds_exp3 = generate_batch(prompts, system_prompt=SYSTEM_PROMPT, enable_thinking=True,
                            thinking_budget=2048, answer_budget=64)
print(f"Прошло времени: {time.time() - t0:.1f}s")

em_exp3 = sum(exact_match(p, g) for p, g in zip(preds_exp3, golds)) / len(golds)
f1_exp3 = sum(f1_lemmas(p, g) for p, g in zip(preds_exp3, golds)) / len(golds)

print(f"\nЭксперимент 2:")
print(f"Exact Match: {em_exp3:.4f}")
print(f"F1 (леммы): {f1_exp3:.4f}")
print(f"\nПример: {preds_exp3[0]}")
print(f"Правильный ответ: {golds[0]}")

Прошло времени: 210.2s

Эксперимент 2:
Exact Match: 0.0000
F1 (леммы): 0.0997

Пример: 1100 years. It had a council of 1100 years. It had a council of 1100 years. It had a council of 1100 years. It had a council of 1100 years. It had a council of 110
Правильный ответ: Большой совет


По результатам 3-х экспериментов можно заметить, что маленькая модель справляется с примерами с промптом и без примерно одинаково хорошо. А вот включенный thinking (неважно, сколько на него отведено токенов, я проверяла и 256, и 512, и 1024) заставляет модель застрять в размышлениях, и не дает вывести нужный пользователю ответ.